In [1]:
F2 = GF(2)
R.<t> = PolynomialRing(F2)
p =t^163 + t^7 + t^6 + t^3 + 1
F163.<x> = GF(2^163, modulus=p)

In [2]:
## Convert the basis of L(D) to elements of F_2^163, from fi() to fi(alpha).
exprs=[]
file_path="2^163_l(2D)_pn_basis.txt"
lines = []
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        lines.append(line.strip())
L2d_basis=[]
for expr_str in lines:
    try:
        expr = eval(expr_str.replace("^", "**"))
        L2d_basis.append(expr)
    except Exception as e:
        print(f"An error occurred while parsing the expression:{e}")
print(len(L2d_basis))

325


In [3]:
from sage.all import matrix
def extract_coefficients(L2d_basis):
    #Get the coefficient matrix
    coeffs_matrix = []
    for expr_str in L2d_basis:
        if(expr_str==1):
            print(expr_str)
            coeffs_full =  [1]+[0] * (162)
            coeffs_matrix.append(coeffs_full)
        else:
            poly = expr_str.polynomial()  
            degree = poly.degree()  
            coeffs = poly.list()  
            coeffs_full = coeffs + [0] * (163 - len(coeffs))
            coeffs_matrix.append(coeffs_full)
    coeffs_matrix = matrix(F2, coeffs_matrix).transpose()

    return coeffs_matrix
l2dbasis_to_xbasis_matrix = extract_coefficients(L2d_basis)
## Obtaining the linear transformation from a polynomial basis to ldbasis
ldbasis_to_xbasis_matrix_163 = l2dbasis_to_xbasis_matrix[:, :163]  
if ldbasis_to_xbasis_matrix_163.is_invertible():
    xbasis_to_ldbasis_matrix = ldbasis_to_xbasis_matrix_163.inverse()  
else:
    print("The matrix is ​​not invertible on F_2.")

In [4]:
def save_matrix_to_file(matrix_obj, filename):
    with open(filename, "w") as f:
        for row in matrix_obj:
            line = "".join(str(x) for x in row)
            f.write(line + "\n")
    print(f"Saved {filename}")

M_L2X = ldbasis_to_xbasis_matrix_163

# Obtain the X -> L matrix (inverse matrix)
# if M_L2X.is_invertible():
#     M_X2L = M_L2X.inverse()
#     save_matrix_to_file(M_L2X, "163_L2X.txt")
#     save_matrix_to_file(M_X2L, "163_X2L.txt")
# else:
#     print("Error: Matrix is not invertible!")

In [5]:
import pickle
import numpy as np

# Reading the matrix
with open("163_TD_pn.pkl", "rb") as f:
    TD = pickle.load(f)
with open("163_CT2D_inv_pn.pkl", "rb") as f:
    CT2D_inv = pickle.load(f)
with open("163_T2D_inv_pn.pkl", "rb") as f:
    T2D_inv = pickle.load(f)

In [6]:
def g_adic_expansion(f_poly, g_poly, num_terms):
    """Calculate the g-adic expansion coefficients of f_poly with respect to g_poly"""
    coeffs_poly = []
    current_f = f_poly
    d=g_poly.degree()
    for _ in range(num_terms):
        c = current_f % g_poly
        coeffs = c.list()
        while len(coeffs) < d:
            coeffs.append(0)
        coeffs_poly.extend(coeffs)
        current_f = (current_f - c) // g_poly
    return coeffs_poly

In [7]:
def mul_2(a, b,m,mvars):
    l=len(m)
    m.append(a[0]*b[0])               # m0: a0*b0
    m.append(a[1]*b[1])               # m1: a1*b1
    m.append((a[0] + a[1]).mod(2)*(b[0] + b[1]).mod(2)) # m2: (a0+a1)(b0+b1)
    return[mvars[l+0],mvars[l+2]-mvars[l+0]-mvars[l+1],mvars[l+1]]
def mul_3(a, b,m,mvars):
    l=len(m)
    m.append(a[0]*b[0])               # m0: a0*b0
    m.append(a[1]*b[1])               # m1: a1*b1
    m.append(a[2]*b[2])            # m2: a2*b2
    m.append((a[0] + a[1]).mod(2)*(b[0] + b[1]).mod(2)) # m3: (a0+a1)(b0+b1)
    m.append((a[1] + a[2]).mod(2)*(b[1] + b[2]).mod(2)) # m4: (a1+a2)(b1+b2)
    m.append((a[0] + a[1] + a[2]).mod(2)*(b[0] + b[1] + b[2]).mod(2))            # m5: (a0+a1+a2)(b0+b1+b2)
    return[mvars[l+0],-1*mvars[l+0] + -1*mvars[l+1] + mvars[l+3],2*mvars[l+1] + -1*mvars[l+3] + -1*mvars[l+4] + mvars[l+5],-1*mvars[l+1] + -1*mvars[l+2] + mvars[l+4],mvars[l+2]]
def mul_4(a, b, m, mvars):
    l = len(m)
    m.append(a[0] * b[0])  # m0
    m.append(a[1] * b[1])  # m1
    m.append(a[2] * b[2])  # m2
    m.append(a[3] * b[3])  # m3
    m.append((a[0] + a[1]).mod(2) * (b[0] + b[1]).mod(2))  # m4
    m.append((a[2] + a[3]).mod(2) * (b[2] + b[3]).mod(2))  # m5
    m.append((a[0] + a[2]).mod(2) * (b[0] + b[2]).mod(2))  # m6
    m.append((a[1] + a[3]).mod(2) * (b[1] + b[3]).mod(2))  # m7
    m.append((a[0] + a[1] + a[2] + a[3]).mod(2) * (b[0] + b[1] + b[2] + b[3]).mod(2))  # m8
    return [mvars[l+0], 
            mvars[l+4] - mvars[l+0] - mvars[l+1],
            mvars[l+6] + mvars[l+1] -mvars[l+0]-mvars[l+2],
            mvars[l+8]+mvars[l+4]  +mvars[l+6]-mvars[l+0]-mvars[l+2]  +mvars[l+7]-mvars[l+1]  +mvars[l+5]-mvars[l+3],
            mvars[l+7] + mvars[l+2] -mvars[l+1]-mvars[l+3],
            mvars[l+5] - mvars[l+2] - mvars[l+3],
            mvars[l+3]
           ]
def mul_5(a, b, m, mvars):
    l = len(m)
    m.append((a[0] + a[1] + a[2] + a[3] + a[4]).mod(2) * (b[0] + b[1] + b[2] + b[3] + b[4]).mod(2))  # m0
    m.append((a[0] - a[2] - a[3] - a[4]).mod(2) * (b[0] - b[2] - b[3] - b[4]).mod(2))  # m1
    m.append((a[0] + a[1] + a[2] - a[4]).mod(2) * (b[0] + b[1] + b[2] - b[4]).mod(2))  # m2
    m.append((a[0] + a[1] - a[3] - a[4]).mod(2) * (b[0] + b[1] - b[3] - b[4]).mod(2))  # m3
    m.append((a[0] - a[2] - a[3]).mod(2) * (b[0] - b[2] - b[3]).mod(2))  # m4
    m.append((a[1] + a[2] - a[4]).mod(2) * (b[1] + b[2] - b[4]).mod(2))  # m5
    m.append((a[3] + a[4]).mod(2) * (b[3] + b[4]).mod(2))  # m6
    m.append((a[0] + a[1]).mod(2) * (b[0] + b[1]).mod(2))  # m7
    m.append((a[0] - a[4]).mod(2) * (b[0] - b[4]).mod(2))  # m8
    m.append(a[4] * b[4])  # m9
    m.append(a[3] * b[3])  # m10
    m.append(a[1] * b[1])  # m11
    m.append(a[0] * b[0])  # m12

    return [
        mvars[l+12],  # X^0
        mvars[l+7] - mvars[l+11] - mvars[l+12],  # X^1
        mvars[l+2] - mvars[l+5] - mvars[l+7] - mvars[l+8] + mvars[l+9] + 2*mvars[l+11] + mvars[l+12],  # X^2
        mvars[l+0] - mvars[l+1] - 2*mvars[l+2] + mvars[l+3] + 2*mvars[l+5] - mvars[l+6] + 3*mvars[l+8] - 3*mvars[l+9] - 2*mvars[l+11] - 2*mvars[l+12],  # X^3
        -mvars[l+0] + 2*mvars[l+1] + 2*mvars[l+2] - 2*mvars[l+3] - mvars[l+4] - mvars[l+5] + mvars[l+6] + mvars[l+7] - 4*mvars[l+8] + 3*mvars[l+9] + mvars[l+10] + mvars[l+11] + 3*mvars[l+12],  # X^4
        mvars[l+0] - 2*mvars[l+1] - mvars[l+2] + mvars[l+3] + 2*mvars[l+4] - mvars[l+7] + 3*mvars[l+8] - 2*mvars[l+9] - 2*mvars[l+10] - 3*mvars[l+12],  # X^5
        mvars[l+1] - mvars[l+4] - mvars[l+6] - mvars[l+8] + mvars[l+9] + 2*mvars[l+10] + mvars[l+12],  # X^6
        mvars[l+6] - mvars[l+9] - mvars[l+10],  # X^7
        mvars[l+9]  # X^8
    ]

def mul_6(a, b, m, mvars):
    l = len(m)
    m.append(a[5]*b[5])  # m1
    m.append(a[2]*b[2])  # m2
    m.append((a[2]+a[4]).mod(2)*(b[2]+b[4]).mod(2))  # m3
    m.append((a[1]+a[5]).mod(2)*(b[1]+b[5]).mod(2))  # m4
    m.append((a[1]+a[3]+a[4]).mod(2)*(b[1]+b[3]+b[4]).mod(2))  # m5
    m.append((a[1]+a[2]+a[5]).mod(2)*(b[1]+b[2]+b[5]).mod(2))  # m6
    m.append((a[1]+a[2]+a[3]+a[5]).mod(2)*(b[1]+b[2]+b[3]+b[5]).mod(2))  # m7
    m.append(a[0]*b[0])  # m8
    m.append((a[0]+a[3]+a[5]).mod(2)*(b[0]+b[3]+b[5]).mod(2))  # m9
    m.append((a[0]+a[2]+a[5]).mod(2)*(b[0]+b[2]+b[5]).mod(2))  # m10
    m.append((a[0]+a[2]+a[3]+a[4]+a[5]).mod(2)*(b[0]+b[2]+b[3]+b[4]+b[5]).mod(2))  # m11
    m.append((a[0]+a[1]+a[2]+a[4]).mod(2)*(b[0]+b[1]+b[2]+b[4]).mod(2))  # m12
    m.append((a[0]+a[1]+a[2]+a[4]+a[5]).mod(2)*(b[0]+b[1]+b[2]+b[4]+b[5]).mod(2))  # m13
    m.append((a[0]+a[1]+a[2]+a[3]+a[5]).mod(2)*(b[0]+b[1]+b[2]+b[3]+b[5]).mod(2))  # m14
    m.append((a[0]+a[1]+a[2]+a[3]+a[4]+a[5]).mod(2)*(b[0]+b[1]+b[2]+b[3]+b[4]+b[5]).mod(2))  # m15
    m1 = mvars[l+0]
    m2 = mvars[l+1]
    m3 = mvars[l+2]
    m4 = mvars[l+3]
    m5 = mvars[l+4]
    m6 = mvars[l+5]
    m7 = mvars[l+6]
    m8 = mvars[l+7]
    m9 = mvars[l+8]
    m10 = mvars[l+9]
    m11 = mvars[l+10]
    m12 = mvars[l+11]
    m13 = mvars[l+12]
    m14 = mvars[l+13]
    m15 = mvars[l+14]

    t1 = m1 + m2 + m3 + m4 + m5 + m8 + m10 + m11 + m13 + m14
    t2 = m3 + m4 + m5 + m6 + m11 + m15
    t3 = m2 + m5 + m6 + m7 + m9 + m11 + m14 + m15
    t4 = m1 + m2 + m5 + m6 + m8 + m9 + m10 + m11 + m12 + m14
    t5 = m1 + m3 + m10 + m11 + m13 + m15
    t6 = m8 + m9 + m10 + m11 + m14 + m15

    return [t1, t2, t3, t4, t5, t6]
def mul_7(a, b, m, mvars):
    l = len(m)
    m.append((a[0] + a[1] + a[2] + a[3] + a[4] + a[5] + a[6]).mod(2) * (b[0] + b[1] + b[2] + b[3] + b[4] + b[5] + b[6]).mod(2))  # m0
    m.append((a[1] + a[2] + a[3] - a[5] - a[6]).mod(2) * (b[1] + b[2] + b[3] - b[5] - b[6]).mod(2))  # m1
    m.append((a[0] + a[1] - a[3] - a[4] - a[5]).mod(2) * (b[0] + b[1] - b[3] - b[4] - b[5]).mod(2))  # m2
    m.append((a[0] - a[2] - a[3] - a[4] + a[6]).mod(2) * (b[0] - b[2] - b[3] - b[4] + b[6]).mod(2))  # m3
    m.append((a[0] - a[2] - a[3] + a[5] + a[6]).mod(2) * (b[0] - b[2] - b[3] + b[5] + b[6]).mod(2))  # m4
    m.append((a[0] + a[1] - a[3] - a[4] + a[6]).mod(2) * (b[0] + b[1] - b[3] - b[4] + b[6]).mod(2))  # m5
    m.append((a[1] + a[2] - a[4] - a[5]).mod(2) * (b[1] + b[2] - b[4] - b[5]).mod(2))  # m6
    m.append((a[0] + a[1] - a[5] - a[6]).mod(2) * (b[0] + b[1] - b[5] - b[6]).mod(2))  # m7
    m.append((a[0] + a[1]).mod(2) * (b[0] + b[1]).mod(2))  # m8
    m.append((a[0] - a[2]).mod(2) * (b[0] - b[2]).mod(2))  # m9
    m.append((a[0] - a[4]).mod(2) * (b[0] - b[4]).mod(2))  # m10
    m.append((a[1] + a[3]).mod(2) * (b[1] + b[3]).mod(2))  # m11
    m.append((a[2] - a[6]).mod(2) * (b[2] - b[6]).mod(2))  # m12
    m.append((a[3] + a[5]).mod(2) * (b[3] + b[5]).mod(2))  # m13
    m.append((a[4] - a[6]).mod(2) * (b[4] - b[6]).mod(2))  # m14
    m.append((a[5] + a[6]).mod(2)* (b[5] + b[6]).mod(2))  # m15
    m.append(a[0] * b[0])  # m16
    m.append(a[1] * b[1])  # m17
    m.append(a[2] * b[2])  # m18
    m.append(a[4] * b[4])  # m19
    m.append(a[5] * b[5])  # m20
    m.append(a[6] * b[6])  # m21
    m0, m1, m2, m3, m4, m5, m6, m7, m8, m9, m10, m11, m12, m13, m14, m15, m16, m17, m18, m19, m20, m21 = [mvars[l+i] for i in range(22)]

    return [
        m16,  # X^0
        m8 - m16 - m17,  # X^1
        -m9 + m16 + m17 + m18,  # X^2
        2*m1 + m2 - m3 - m4 - m6 - 2*m7 + m8 + 2*m9 - m11 + m14 + m15 - m18,  # X^3
        -m1 - m2 + m3 + m6 + m7 - m9 - m10 + m11 - m14 + m16 - m17 + m18 + m19,  # X^4
        m0 - 3*m1 - 3*m2 + m3 + 2*m4 + m5 + 2*m6 + 5*m7 - 4*m8 - 2*m9 + 2*m10 + m12 + m13 - m14 - 5*m15 - 3*m16 + m17 - 2*m18 - 3*m19 - 2*m21,  # X^5
        -m0 + 5*m1 + 5*m2 - 2*m3 - 2*m4 - 2*m5 - 4*m6 - 8*m7 + 6*m8 + 3*m9 - 2*m10 - m11 - 2*m12 - m13 + 3*m14 + 6*m15 + 3*m16 + 3*m18 + 3*m19 + 3*m21,  # X^6
        m0 - 3*m1 - 3*m2 + m3 + m4 + 2*m5 + 2*m6 + 5*m7 - 5*m8 - m9 + m10 + m11 + 2*m12 - 2*m14 - 4*m15 - 2*m16 - 3*m18 - 2*m19 + m20 - 3*m21,  # X^7
        -m1 - m2 + m3 + m6 + m7 - m9 - m12 + m13 - m14 + m18 + m19 - m20 + m21,  # X^8
        m1 + 2*m2 - m3 - m5 - m6 - 2*m7 + m8 + m9 - m13 + 2*m14 + m15 - m19,  # X^9
        -m14 + m19 + m20 + m21,  # X^10
        m15 - m20 - m21,  # X^11
        m21  # X^12
    ]
def mul_8(a, b, m, mvars):
    l = len(m)
    m.append((a[0] + a[3] + a[4] + a[7]).mod(2) * (b[0] + b[3] + b[4] + b[7]).mod(2))  # m0
    m.append((a[0] + a[1] + a[5]).mod(2) * (b[0] + b[1] + b[5]).mod(2))                # m1
    m.append((a[0] + a[1] + a[6] + a[7]).mod(2) * (b[0] + b[1] + b[6] + b[7]).mod(2))  # m2
    m.append((a[0] + a[1] + a[2] + a[4] + a[5] + a[6]).mod(2) * (b[0] + b[1] + b[2] + b[4] + b[5] + b[6]).mod(2))  # m3
    m.append((a[0] + a[1] + a[2] + a[4] + a[7]).mod(2) * (b[0] + b[1] + b[2] + b[4] + b[7]).mod(2))                # m4
    m.append((a[0] + a[2] + a[3] + a[4] + a[5] + a[6] + a[7]).mod(2) * (b[0] + b[2] + b[3] + b[4] + b[5] + b[6] + b[7]).mod(2))  # m5
    m.append((a[0] + a[2] + a[5] + a[6] + a[7]).mod(2) * (b[0] + b[2] + b[5] + b[6] + b[7]).mod(2))                # m6
    m.append((a[0] + a[1] + a[2] + a[3] + a[4] + a[5] + a[6] + a[7]).mod(2) * (b[0] + b[1] + b[2] + b[3] + b[4] + b[5] + b[6] + b[7]).mod(2))  # m7
    m.append((a[1] + a[2] + a[4] + a[5]).mod(2) * (b[1] + b[2] + b[4] + b[5]).mod(2))                            # m8
    m.append((a[2] + a[5] + a[7]).mod(2) * (b[2] + b[5] + b[7]).mod(2))                                          # m9
    m.append((a[2] + a[3] + a[4] + a[6]).mod(2) * (b[2] + b[3] + b[4] + b[6]).mod(2))                            # m10
    m.append((a[1] + a[4]).mod(2) * (b[1] + b[4]).mod(2))                                                        # m11
    m.append((a[1] + a[3] + a[5] + a[6] + a[7]).mod(2) * (b[1] + b[3] + b[5] + b[6] + b[7]).mod(2))              # m12
    m.append((a[1] + a[3] + a[4] + a[6] + a[7]).mod(2) * (b[1] + b[3] + b[4] + b[6] + b[7]).mod(2))              # m13
    m.append((a[1] + a[3] + a[4] + a[5]).mod(2) * (b[1] + b[3] + b[4] + b[5]).mod(2))                            # m14
    m.append((a[1] + a[2] + a[4] + a[5] + a[6]).mod(2) * (b[1] + b[2] + b[4] + b[5] + b[6]).mod(2))              # m15
    m.append((a[0] + a[1] + a[2] + a[3] + 2*a[4] + a[5] + a[7]).mod(2) * (b[0] + b[1] + b[2] + b[3] + 2*b[4] + b[5] + b[7]).mod(2))  # m16
    m.append((a[0] + a[1] + a[2] + 2*a[5] + a[7]).mod(2) * (b[0] + b[1] + b[2] + 2*b[5] + b[7]).mod(2))  # m17
    m.append((a[0] + a[1] + a[2] + a[3] + a[4] + 2*a[6] + a[7]).mod(2) * (b[0] + b[1] + b[2] + b[3] + b[4] + 2*b[6] + b[7]).mod(2))  # m18
    m.append((a[0] + 2*a[1] + a[2] + 2*a[4] + a[5] + a[6]).mod(2) * (b[0] + 2*b[1] + b[2] + 2*b[4] + b[5] + b[6]).mod(2))  # m19
    m.append((a[0] + 2*a[1] + a[2] + a[3] + a[4] + a[5] + a[6] + 2*a[7]).mod(2) * (b[0] + 2*b[1] + b[2] + b[3] + b[4] + b[5] + b[6] + 2*b[7]).mod(2))  # m20
    m.append((a[0] + a[1] + a[2] + 2*a[3] + 2*a[4] + a[5] + 2*a[6] + 2*a[7]).mod(2) * (b[0] + b[1] + b[2] + 2*b[3] + 2*b[4] + b[5] + 2*b[6] + 2*b[7]).mod(2))  # m21
    m.append((a[0] + a[1] + a[2] + a[3] + a[4] + 2*a[5] + a[6] + a[7]).mod(2) * (b[0] + b[1] + b[2] + b[3] + b[4] + 2*b[5] + b[6] + b[7]).mod(2))  # m22
    m.append((a[0] + 2*a[1] + 2*a[2] + a[3] + 2*a[4] + 2*a[5] + 2*a[6] + a[7]).mod(2) * (b[0] + 2*b[1] + 2*b[2] + b[3] + 2*b[4] + 2*b[5] + 2*b[6] + b[7]).mod(2))  # m23

    return [
        mvars[l+0] + mvars[l+11] + mvars[l+12] + mvars[l+14] + mvars[l+17] + mvars[l+18] - mvars[l+2] + mvars[l+20] + mvars[l+21] + mvars[l+23] + mvars[l+3] - mvars[l+5] + mvars[l+6] - mvars[l+7] + mvars[l+8] + mvars[l+9],
        -mvars[l+0] + mvars[l+15] + mvars[l+16] + mvars[l+17] + mvars[l+22] + mvars[l+23] - mvars[l+6] + mvars[l+9],
        mvars[l+0] + mvars[l+10] + mvars[l+14] + mvars[l+15] + mvars[l+18] + mvars[l+19] + mvars[l+20] + mvars[l+21] + mvars[l+22] - mvars[l+3] - mvars[l+4] - mvars[l+5] + mvars[l+7] + mvars[l+8],
        mvars[l+11] + mvars[l+12] + mvars[l+18] + mvars[l+19] - mvars[l+2] + mvars[l+20] + mvars[l+22] + mvars[l+23] - mvars[l+6] - mvars[l+7],
        -mvars[l+0] + mvars[l+10] + mvars[l+12] + mvars[l+13] + mvars[l+14] + mvars[l+15] + mvars[l+16] + mvars[l+17] + mvars[l+19] + mvars[l+2] - mvars[l+3] + mvars[l+4] + mvars[l+5] + mvars[l+6] + mvars[l+7] + mvars[l+9],
        mvars[l+0] + mvars[l+10] + mvars[l+11] + mvars[l+12] + mvars[l+15] + mvars[l+17] + mvars[l+18] + mvars[l+19] + mvars[l+21] + mvars[l+23] + mvars[l+4] - mvars[l+5] + mvars[l+8] + mvars[l+9],
        mvars[l+0] + mvars[l+11] + mvars[l+14] + mvars[l+15] + mvars[l+19] + mvars[l+20] + mvars[l+22] + mvars[l+23] - mvars[l+4] + mvars[l+8],
        -mvars[l+1] + mvars[l+15] + mvars[l+16] + mvars[l+17] + mvars[l+18] + mvars[l+19] - mvars[l+2] + mvars[l+22] - mvars[l+3] - mvars[l+6] + mvars[l+7] + mvars[l+8]
    ]

In [8]:
def Mul(A, a_syms):
    n = A.nrows()
    m = A.ncols()
    assert len(a_syms) == m
    result = []
    for i in range(n):
        expr = 0
        for j in range(m):
            if A[i, j] == 1:
                expr += a_syms[j]
        try:
            expr = expr.mod(2)
        except Exception:
            expr = expr % 2
        result.append(expr)
    return result
def find_all_combinations(a):
    comb = []
    max_s = (a - 1) // 2
    for s in range(0, max_s + 1):
        t = a - s
        if t > s:  
            comb.append((s, t))
    return comb
def compute_modulo_matrix(field_degree, g):
    F = g.base_ring()
    R = g.parent()
    x = R.gen()
    modulo_matrix = matrix(F, field_degree, 2 * field_degree-1)
    for i in range(2 * field_degree-1):
        f = x**i
        remainder = f % g
        coeffs = remainder.coefficients(sparse=False)
        coeffs += [F(0)] * (field_degree - len(coeffs))
        for j in range(field_degree):
            modulo_matrix[j, i] = coeffs[j]
    return modulo_matrix

In [9]:
F2 = GF(2)
R.<x> = PolynomialRing(F2) 
mod_polys = [x, x + 1, x^2 + x + 1, x^3 + x + 1, x^3 + x^2 + 1, x^4 + x + 1, x^4 + x^3 + 1, x^4 + x^3 + x^2 + x + 1, x^5 + x^2 + 1, x^5 + x^3 + 1, x^5 + x^3 + x^2 + x + 1, x^5 + x^4 + x^2 + x + 1, x^5 + x^4 + x^3 + x + 1, x^5 + x^4 + x^3 + x^2 + 1, x^6 + x + 1, x^6 + x^3 + 1, x^6 + x^4 + x^2 + x + 1, x^6 + x^4 + x^3 + x + 1, x^6 + x^5 + 1, x^6 + x^5 + x^2 + x + 1, x^6 + x^5 + x^3 + x^2 + 1, x^6 + x^5 + x^4 + x + 1, x^6 + x^5 + x^4 + x^2 + 1, x^8 + x^4 + x^3 + x + 1, x^8 + x^4 + x^3 + x^2 + 1, x^8 + x^5 + x^3 + x + 1, x^8 + x^5 + x^3 + x^2 + 1, x^8 + x^5 + x^4 + x^3 + 1, x^8 + x^5 + x^4 + x^3 + x^2 + x + 1, x^8 + x^6 + x^3 + x^2 + 1, x^8 + x^6 + x^4 + x^3 + x^2 + x + 1, x^8 + x^6 + x^5 + x + 1, x^8 + x^6 + x^5 + x^2 + 1, x^8 + x^6 + x^5 + x^3 + 1, x^8 + x^6 + x^5 + x^4 + 1, x^8 + x^6 + x^5 + x^4 + x^2 + x + 1, x^8 + x^6 + x^5 + x^4 + x^3 + x + 1, x^8 + x^7 + x^2 + x + 1, x^8 + x^7 + x^3 + x + 1, x^8 + x^7 + x^3 + x^2 + 1, x^8 + x^7 + x^4 + x^3 + x^2 + x + 1, x^8 + x^7 + x^5 + x + 1, x^8 + x^7 + x^5 + x^3 + 1, x^8 + x^7 + x^5 + x^4 + 1, x^8 + x^7 + x^5 + x^4 + x^3 + x^2 + 1, x^8 + x^7 + x^6 + x + 1, x^8 + x^7 + x^6 + x^3 + x^2 + x + 1, x^8 + x^7 + x^6 + x^4 + x^2 + x + 1, x^8 + x^7 + x^6 + x^4 + x^3 + x^2 + 1]

In [10]:
F2 = GF(2)
R.<x> = PolynomialRing(F2)
def change_basis(g):
    F2 = GF(2)
    R.<x> = PolynomialRing(F2)
    F1 = GF(2^8, name='a', modulus=g, impl='pari')
    a = F1.gen()
    F2_ext = GF(2^8, name='b', modulus=x^8 + x^4 + x^3 + x + 1, impl='pari')
    b = F2_ext.gen()
    minpoly_a = a.minpoly()
    for beta in F2_ext:
        if beta == 0:
            continue
        if beta.minpoly() == minpoly_a:
            break
    else:
        raise ValueError("No valid mapping found")
    M = Matrix(F2, 8, 8)
    for i in range(8):
        v = a^i
        v2 = beta^i
        poly = v2.polynomial()
        coeffs = poly.coefficients(sparse=False)
        coeffs += [0] * (8 - len(coeffs))
        M.set_column(i, vector(F2, coeffs))
    return M
def change_basis_6(g):
    F2 = GF(2)
    R.<x> = PolynomialRing(F2)
    F1 = GF(2^6, name='a', modulus=g, impl='pari')
    a = F1.gen()
    F2_ext = GF(2^6, name='b', modulus=x^6 + x + 1, impl='pari')
    b = F2_ext.gen()
    minpoly_a = a.minpoly()
    for beta in F2_ext:
        if beta == 0:
            continue
        if beta.minpoly() == minpoly_a:
            break
    else:
        raise ValueError("No valid mapping found")
    M = Matrix(F2, 6, 6)
    for i in range(6):
        v = a^i
        v2 = beta^i
        poly = v2.polynomial()
        coeffs = poly.coefficients(sparse=False)
        coeffs += [0] * (6 - len(coeffs))
        M.set_column(i, vector(F2, coeffs))
    return M

In [11]:
def polynomial_multiply_first_t_coeffs(a, b, deg,u,m,mvars,result,mod_poly):
    combs = []
    for i in range(u-1):
        comb = find_all_combinations(i+1)
        combs.extend(comb)
    combs = list(set(combs))
    D_st={}
    for [idx1,idx2] in combs:
        key = (idx1, idx2)
        a_idx=[]
        b_idx=[]
        for i in range(deg):
            a_idx.append(a[deg*idx1+i]+a[deg*idx2+i])
            b_idx.append(b[deg*idx1+i]+b[deg*idx2+i])
        if(deg==1):
            l=len(m)
            m.append((a[idx1]+a[idx2])*(b[idx1]+b[idx2]))
            D_st[key] = [mvars[l]]  
        elif(deg==2):
            modulo_2=compute_modulo_matrix(2, mod_poly)
            res=Mul(modulo_2,mul_2(a_idx,b_idx,m,mvars))
            D_st[key] =(res)
        elif(deg==3):
            modulo_3=compute_modulo_matrix(3, mod_poly)
            res=Mul(modulo_3,mul_3(a_idx,b_idx,m,mvars))
            D_st[key] =(res)
        elif(deg==4):
            modulo_4=compute_modulo_matrix(4, mod_poly)
            res=Mul(modulo_4,mul_4(a_idx,b_idx,m,mvars))
            D_st[key] =(res)
        elif(deg==5):
            modulo_5=compute_modulo_matrix(5, mod_poly)
            res=Mul(modulo_5,mul_5(a_idx,b_idx,m,mvars))
            D_st[key] =(res)
        elif(deg==6):
            modulo_6=change_basis_6(mod_poly)
            new_a_P = Mul(modulo_6,a_idx)
            new_b_P = Mul(modulo_6,b_idx)
            mul_result =mul_6(new_a_P,new_b_P,m,mvars)
            D_st[key] =Mul(modulo_6.inverse(),mul_result)
        elif(deg==7):
            modulo_7=compute_modulo_matrix(7, mod_poly)
            res=Mul(modulo_7,mul_7(a_idx,b_idx,m,mvars))
            D_st[key] =(res)
        elif(deg==8):
            modulo_8=change_basis(mod_poly)
            new_a_P = Mul(modulo_8,a_idx)
            new_b_P = Mul(modulo_8,b_idx)
            mul_result =mul_8(new_a_P,new_b_P,m,mvars)
            D_st[key]=Mul(modulo_8.inverse(),mul_result)
    D_i = {}
    for i in range(u):
        if(deg==1):
            l=len(m)
            m.append((a[i])*(b[i]))
            D_i[i] = [mvars[l]]  
        elif(deg==2):
            modulo_2=compute_modulo_matrix(2, mod_poly)
            res=Mul(modulo_2,mul_2(a[deg*i:deg*(i+1)], b[deg*i:deg*(i+1)],m,mvars))
            D_i[i]=(res)
        elif(deg==3):
            modulo_3=compute_modulo_matrix(3, mod_poly)
            res=Mul(modulo_3,mul_3(a[deg*i:deg*(i+1)], b[deg*i:deg*(i+1)],m,mvars))
            D_i[i]=(res)
        elif(deg==4):
            modulo_4=compute_modulo_matrix(4, mod_poly)
            res=Mul(modulo_4,mul_4(a[deg*i:deg*(i+1)], b[deg*i:deg*(i+1)],m,mvars))
            D_i[i]=(res)
        elif(deg==5):
            modulo_5=compute_modulo_matrix(5, mod_poly)
            res=Mul(modulo_5,mul_5(a[deg*i:deg*(i+1)], b[deg*i:deg*(i+1)],m,mvars))
            D_i[i]=(res)
        elif(deg==6):
            modulo_6=change_basis_6(mod_poly)
            new_a_P = Mul(modulo_6,a[deg*i:deg*(i+1)])
            new_b_P = Mul(modulo_6,b[deg*i:deg*(i+1)])
            mul_result =mul_6(new_a_P,new_b_P,m,mvars)
            D_i[i]=Mul(modulo_6.inverse(),mul_result)
        elif(deg==7):
            modulo_7=compute_modulo_matrix(7, mod_poly)
            res=Mul(modulo_7,mul_7(a[deg*i:deg*(i+1)], b[deg*i:deg*(i+1)],m,mvars))
            D_i[i]=(res)
        elif(deg==8):
            modulo_8=change_basis(mod_poly)
            new_a_P = Mul(modulo_8,a[deg*i:deg*(i+1)])
            new_b_P = Mul(modulo_8,b[deg*i:deg*(i+1)])
            mul_result =mul_8(new_a_P,new_b_P,m,mvars)
            D_i[i]=Mul(modulo_8.inverse(),mul_result)
    result.extend([D_i[0]])
    for i in range(u-1):
        comb = find_all_combinations(i+1)
        res=[0]*deg
        for (idx1,idx2) in comb:
            for j in range(deg):
                res[j]+=D_st[(idx1,idx2)][j]
                res[j]+=D_i[idx1][j]
                res[j]+=D_i[idx2][j]
        if (i+1) % 2 == 0:
            if deg == 1:
                res[0] += D_i[(i+1)//2][0]
            else:
                for j in range(deg):
                    res[j] += D_i[(i+1)//2][j]
            result.extend(res)
        else:
            result.extend(res)

    return result

In [12]:
def find_all_combinations_w(a,n):
    comb = []
    for s in range(0, n):
        t = a - s
        if (t > s)&(t< n):  
            comb.append((s, t))
    return comb
def compute_highest_three_terms(a_p, b_p, m, mvars,u):
    """
    Calculate the coefficient of the highest u term in a polynomial product.
    """
    result=[]
    n=len(a_p)
    combs=[]
    for i in range(u-1):
        comb = find_all_combinations_w(2*n-2-(i+1),n)
        combs.extend(comb)
    combs = list(set(combs))
    D_st={}
    for [idx1,idx2] in combs:
        key = (idx1, idx2)
        l=len(m)
        m.append((a_p[idx1]+a_p[idx2])*(b_p[idx1]+b_p[idx2]))
        D_st[key] = mvars[l]
    D_i = {}
    
    for i in range(u):
        l=len(m)
        m.append((a_p[n-1-i])*(b_p[n-1-i]))
        D_i[n-1-i] = mvars[l]
    
    result.extend([D_i[n-1]])
    for i in range(u-1):
        comb = find_all_combinations_w(2*n-2-(i+1),n)
        res=0
        for (idx1,idx2) in comb:
            res+=D_st[(idx1,idx2)]
        for j in range(i+2): 
            res+=D_i[n-1-(j)]
        result.extend([res])

    return result[::-1]


In [13]:
def mul_smallfield(a,b,m,mvars,u,num,result):
    idx=0
    sum=0
    for i in range(len(num)):
        for j in range(num[i]):
            result=polynomial_multiply_first_t_coeffs(a[idx:idx+(i+1)*u[i]], b[idx:idx+(i+1)*u[i]], i+1,u[i],m,mvars,result,mod_polys[sum+j])
            idx+=(i+1)*u[i]
        sum+=num[i]
    result = [item for sublist in result for item in (sublist if isinstance(sublist, list) else [sublist])]
    result.extend(compute_highest_three_terms(a, b, m, mvars,u[-1]))
    return result

In [14]:
## Construct a coefficient matrix, where each column represents the coefficients of L2d_basis[163+i] under the first 163 bases.
from sage.all import Matrix, GF, identity_matrix
F2 = GF(2)
coeffs_matrix = []
for i in range(162):
    coeffs = l2dbasis_to_xbasis_matrix[:,:163].solve_right(l2dbasis_to_xbasis_matrix[:,163+i])
    coeffs_list=[]
    for j in range(163):
        coeffs_list.append(coeffs[j][0])
    coeffs_matrix.append(coeffs_list)  
coeffs_matrix = Matrix(F2, coeffs_matrix).transpose()
identity_163 = identity_matrix(F2, 163)
coeffs_matrix = identity_163.augment(coeffs_matrix)

In [15]:
from sympy import symbols
from collections import OrderedDict
from sage.all import var
a_syms = var(['a%d' % i for i in range(163)])
b_syms = var(['b%d' % i for i in range(163)])
a_p=Mul(TD*xbasis_to_ldbasis_matrix, a_syms)
b_p=Mul(TD*xbasis_to_ldbasis_matrix, b_syms)
mvars = var(['m%d' % i for i in range(906)])
x = symbols('x')
m=[]
result=[]
u=[5, 1, 1, 1, 1, 1, 0, 1, 3]
num=[2, 1, 2, 3, 6, 9,0, 26]
idx=0
sum=0
result=mul_smallfield(a_p,b_p,m,mvars,u,num,result)
c_ldbasis=Mul(CT2D_inv,result)
dd=Mul(ldbasis_to_xbasis_matrix_163,c_ldbasis)

In [16]:
import re
from sage.all import SR
def mod2_expr(expr):
    def repl(m):
        coef = m.group(1)
        var = m.group(2)
        idx = m.group(3)
        if coef and int(coef) % 2 == 0:
            return ''
        return f'{var}{idx}'
    expr = re.sub(r'(\d+)\s*\*\s*([ab])(\d+)', repl, expr)
    expr = re.sub(r'(\d*)\s*([ab])(\d+)', repl, expr)
    terms = re.findall(r'[+-]?\s*[ab]\d+', expr)
    expr = '+'.join(t.replace(' ', '') for t in terms if t.strip())
    expr = re.sub(r'\++', '+', expr)
    expr = re.sub(r'\-+', '-', expr)
    expr = re.sub(r'\+\-', '-', expr)
    expr = re.sub(r'\-\+', '-', expr)
    expr = re.sub(r'\+\)', ')', expr)
    expr = re.sub(r'\(\+', '(', expr)
    expr = re.sub(r'^\+', '', expr)
    expr = re.sub(r'^\-', '-', expr)
    expr = re.sub(r'\+\Z', '', expr)
    expr = re.sub(r'\-\Z', '', expr)
    return expr

for idx, mi in enumerate(m):
    mi_str = str(mi)
    new_line = re.sub(r'\((.*?)\)', lambda m: '(' + mod2_expr(m.group(1)) + ')', mi_str)
    m[idx] = SR(new_line) 

def mod2_simplify(expr):
    expr = SR(expr).expand()
    m_vars = [v for v in expr.variables() if str(v).startswith('m')]
    result = 0
    for v in m_vars:
        coef = expr.coefficient(v)
        coef_mod2 = ZZ(coef) % 2
        if coef_mod2 == 1:
            result += v
    const = expr.subs({v:0 for v in m_vars})
    const_mod2 = ZZ(const) % 2
    if const_mod2 == 1:
        result += 1
    return result

def flatten(lst):
    for item in lst:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item

dd_flat = list(flatten(dd))
dd_mod2 = [mod2_simplify(e) for e in dd_flat]

In [17]:
import random
a_values =  [random.randint(0, 1) for _ in range(163)]
b_values = [random.randint(0, 1) for _ in range(163)]
a_values =  [0]*162+[1]
b_values = [0]*162+[1]
max_idx1 = max(i for i, v in enumerate(b_values) if v != 0)
max_idx2 = max(i for i, v in enumerate(b_values) if v != 0)
subs_dict = {}
for i in range(163):
    subs_dict[a_syms[i]] = a_values[i]
    subs_dict[b_syms[i]] = b_values[i]

m_num = [mi.subs(subs_dict) for mi in m]

for i in range(len(m)):
    subs_dict[var('m%d' % i)] = m_num[i]

dd_evaluated = [ZZ(expr.subs(subs_dict)) % 2 for expr in dd_mod2]
print(dd_evaluated)
not_one_positions = [i for i, v in enumerate(dd_evaluated) if v == 1]
print("The position in outs that is not 1:", not_one_positions)


F2 = GF(2)
R.<t> = PolynomialRing(F2)
p = t^163 + t^7 + t^6 + t^3 + 1
F163.<x> = GF(2^163, modulus=p)

poly_a = F163(0)
for i in range(163):
    poly_a += a_values[i] * x^i

poly_b = F163(0)
for i in range(163):
    poly_b += b_values[i] * x^i
    
A = F163(poly_a)
B = F163(poly_b)
C = A * B
poly = C.polynomial() 
coeffs = C.polynomial().coefficients(sparse=False)
coeffs += [0]*(163-len(coeffs)) 

c = vector(F2, coeffs)
print("Result of actual field multiplication:", c)

dd_evaluated_vector = vector(F2, dd_evaluated)
diff_vector = c - dd_evaluated_vector
if any(diff_vector):
    print("❌ The difference vector contains an element of 1, indicating an inconsistency!")
    print(diff_vector)
else:
    print("✅ The difference vectors are all 0, they are completely identical!")

[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
The position in outs that is not 1: [1, 5, 10, 12, 161]
Result of actual field multiplication: (0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [18]:
for test_round in range(400):
    try:
        print(test_round)
        a_values = [random.randint(0, 1) for _ in range(163)]
        b_values = [random.randint(0, 1) for _ in range(163)]
        subs_dict = {}
        for i in range(163):
            subs_dict[a_syms[i]] = a_values[i]
            subs_dict[b_syms[i]] = b_values[i]
        m_num = [mi.subs(subs_dict) for mi in m]
        for i in range(len(m)):
            subs_dict[var('m%d' % i)] = m_num[i]
        dd_evaluated = [ZZ(expr.subs(subs_dict)) % 2 for expr in dd_mod2]

        F2 = GF(2)
        R.<t> = PolynomialRing(F2)
        p = t^163 + t^7 + t^6 + t^3 + 1
        F163.<x> = GF(2^163, modulus=p)
        poly_a = F163(0)
        for i in range(163):
            poly_a += a_values[i] * x^i
        poly_b = F163(0)
        for i in range(163):
            poly_b += b_values[i] * x^i
        A = F163(poly_a)
        B = F163(poly_b)
        C = A * B
        coeffs = C.polynomial().coefficients(sparse=False)
        coeffs += [0] * (163 - len(coeffs))
        c = vector(F2, coeffs)
        dd_evaluated_vector = vector(F2, dd_evaluated)
        diff_vector = c - dd_evaluated_vector
        if any(diff_vector):
            print(f"The {test_round+1}th test failed; the difference vector contains an element of 1!")
            print(diff_vector)
            break
    except Exception as e:
        print(f"An exception occurred during the {test_round+1}th test: {e}")
        break
else:
    print("All tests passed!")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27


KeyboardInterrupt: 

In [ ]:
# with open("mul_2.txt", "w", encoding="utf-8") as f:
#     for expr in m:
#         f.write(str(expr) + "\n")
# with open("mul_1.txt", "w", encoding="utf-8") as f:
#     for expr in dd_mod2:
#         f.write(str(expr) + "\n")